# Q2.9 Weight Init and Symmetry

In [5]:
import sys
from pathlib import Path
import argparse
import numpy as np
import wandb
from sklearn.metrics import precision_recall_fscore_support

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.data_loader import load_dataset
from src.ann.neural_network import NeuralNetwork

ENTITY = 'anandhakrishnanm21-indian-institute-of-technology-madras'


In [6]:
def to_namespace(cfg):
    d = dict(cfg)
    defaults = {
        'hidden_size': [128, 64],
        'activation': 'relu',
        'weight_init': 'xavier',
        'loss': 'cross_entropy',
        'learning_rate': 1e-3,
        'weight_decay': 0.0,
        'optimizer': 'sgd',
        'epochs': 10,
        'batch_size': 64,
    }
    defaults.update(d)
    defaults['num_layers'] = len(defaults['hidden_size'])
    return argparse.Namespace(**defaults)

def evaluate_split(model, X, Y):
    logits = model.forward(X)
    loss, acc = model.evaluate(X, Y)
    y_true = np.argmax(Y, axis=1)
    y_pred = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        'loss': float(loss),
        'acc': float(acc),
        'precision': float(p),
        'recall': float(r),
        'f1': float(f1),
        'y_true': y_true,
        'y_pred': y_pred,
    }

In [8]:
PROJECT = 'q2_9_weight_init_symmetry'



def run_weight_init_experiment(weight_init='zeros', steps=50):
    cfg = {
        'epochs': 1, 'batch_size': 64, 'learning_rate': 1e-2, 'weight_decay': 0.0,
        'optimizer': 'sgd', 'activation': 'relu', 'hidden_size': [128, 128],
        'weight_init': weight_init, 'loss': 'cross_entropy'
    }
    run = wandb.init(project=PROJECT, name=f'init_{weight_init}', config=cfg, reinit=True)
    args = to_namespace(wandb.config)
    model = NeuralNetwork(args)
    Xtr, Ytr, Xva, Yva, *_ = load_dataset('mnist')

    step = 0
    for start in range(0, Xtr.shape[0], args.batch_size):
        if step >= steps:
            break
        xb = Xtr[start:start+args.batch_size]
        yb = Ytr[start:start+args.batch_size]
        logits = model.forward(xb)
        model.backward(yb, logits)
        gw = model.layers[0].grad_W
        model.update_weights()

        tr = evaluate_split(model, Xtr, Ytr)
        va = evaluate_split(model, Xva, Yva)

        neuron_norms = {f'neuron_{i-9}': float(np.linalg.norm(gw[:, i])) for i in range(10, 15)}
        payload = {
            'epoch': step + 1,
            'train_loss': tr['loss'], 'train_acc': tr['acc'], 
            'val_loss': va['loss'], 'val_acc': va['acc'],
            **neuron_norms
        }
        wandb.log(payload)
        
        step += 1
    wandb.finish()

run_weight_init_experiment('zeros')
run_weight_init_experiment('xavier')


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
neuron_1,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_2,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_3,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_4,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
neuron_5,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,████████████████████▁▁██▁▁▁█████████████
train_loss,██▇▇▆▇▇▆▆▆▅▅▅▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▂▂▂▃▂▂▂▂▁▁
val_acc,███████████████████▁██▁▁▁▁██████████████
val_loss,██▇▇▆▇▇▆▆▆▅▅▅▅▅▅▅▅▅▅▅▅▄▄▅▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
epoch,50


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
neuron_1,▄▄▂▂▃▇▄▅▁▄▅▆▆▄▄▄█▇▃▅▁▅▄▃▅▃▃█▄▃▃▃▅▄▃▅▂▃▃▃
neuron_2,▃▁▂▃▁▂▃▂▂▂▂▃▃▃▂▆▂▆█▄▂▆▃▅▄▃▃▄▅▂▁▅▇▂▄▄▅▃▂▅
neuron_3,▄▆▂▄▄▄▅▁▂█▇▃▇▇▂▂▂▄▃▃▄▂▁▇▁▁▄▄▄▂▄▂▂▄▃▂▂▂▅▂
neuron_4,▇▄▄▄▄▆▄█▄▄▄▄▄▆▃▃▃▄▅▇▆▅▄▅▃▄▂▇▄▅▆▅▆▃▄▁▆▅▄▃
neuron_5,▅▄▄▁▂▄▄▁▄▆▃▁▅▆▆▅▂▂▇▂▂▄▃▄▃▇▁▁▅▃▇▃▄▂▃▆▂▇█▄
train_acc,▁▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
train_loss,███▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
val_acc,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇███
val_loss,███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
epoch,50
